In [2]:
import pandas as pd
import sqlite3
from pathlib import Path

project_root = Path.cwd().parent
raw_data = project_root / "data" / "raw" / "customer_support_tickets.csv"
db = project_root / "data" / "processed" / "customer_support.db"

In [6]:
query = '''
SELECT 
    c.customer_name,
    p.product_name,
    tt.ticket_type_name,
    t.ticket_priority, 
    t.ticket_channel, 
    t.ticket_status, 
    t.customer_satisfaction_rating, 
    c.customer_age, 
    c.customer_gender, 
    t.first_response_time, 
    t.time_to_resolution, 
    t.ticket_description
FROM tickets t
JOIN customers c ON c.customer_id = t.customer_id
JOIN products p ON p.product_id = t.product_id
JOIN ticket_types tt ON tt.ticket_type_id = t.ticket_type_id;
'''

with sqlite3.connect(db) as conn:
    table = pd.read_sql(query, conn)

table["first_response_time"] = pd.to_datetime(table["first_response_time"])
table["time_to_resolution"] = pd.to_datetime(table["time_to_resolution"])
table.info()

<class 'pandas.DataFrame'>
RangeIndex: 8469 entries, 0 to 8468
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   customer_name                 8469 non-null   str           
 1   product_name                  8469 non-null   str           
 2   ticket_type_name              8469 non-null   str           
 3   ticket_priority               8469 non-null   str           
 4   ticket_channel                8469 non-null   str           
 5   ticket_status                 8469 non-null   str           
 6   customer_satisfaction_rating  2769 non-null   float64       
 7   customer_age                  8469 non-null   int64         
 8   customer_gender               8469 non-null   str           
 9   first_response_time           5650 non-null   datetime64[us]
 10  time_to_resolution            2769 non-null   datetime64[us]
 11  ticket_description            8469 non-nu

In [7]:
table["resolution_time_hours"] = table["time_to_resolution"] - table["first_response_time"] 
table["resolution_time_hours"] = table["resolution_time_hours"].dt.total_seconds() / 3600
table["resolution_time_hours"].describe()

count    2769.000000
mean       -0.057704
std         9.564112
min       -23.233333
25%        -6.933333
50%         0.166667
75%         6.483333
max        23.466667
Name: resolution_time_hours, dtype: float64

### Вывод
Такой разброс значений показывает, что часть ответов на запросы были раньше самих запросов.
Значит значения для синтетического датасета были сгенерированы отдельно друг от друга, не буду на них опираться